In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

### That alter dataset showed coupling I really want to see if it works with the unseen pairs.

In [21]:
import os
import torch
from utils.load_pipeline import GenerateEvulatePairs
from torch.utils.data import DataLoader, random_split
from script.model_dim_4_layer_3 import FibonacciModDatasetSmallShample, model, MinimalTransformer, evaluate_model, train_model, train_plot, eval_plot


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



checkpoint_dir = '../checkpoints'
full_path = os.path.join(checkpoint_dir, 'dim_4_layer_3_alterset.pth')

vocab_size = 10
model = MinimalTransformer(vocab_size=vocab_size).to(device=device)

# model.load_state_dict(
#     torch.load(full_path, map_location=device)['model_state_dict']
# )



# batch_size = 8

# generated_ds = FibonacciModDatasetSmallShample(mod=vocab_size, seq_len=20)

# train_size = int(0.8 * len(generated_ds))
# test_size = len(generated_ds) - train_size
# train_ds, test_ds = random_split(generated_ds, [train_size, test_size])

# train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
# test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)

# avg_loss, eval_accuracy  = evaluate_model(model, test_loader, show_accuracy=True)

# unseen = GenerateEvulatePairs(dataset=train_ds, mod=vocab_size)
# if len(unseen) == 0:
#     print("No unseen data!")



In [24]:
import random
vocab_size = 10
batch_size = 32
seq_len = 20


all_seeds = [(i, j) for i in range(vocab_size) for j in range(vocab_size)]
random.shuffle(all_seeds)

split = int(0.6 * len(all_seeds))
train_seeds = all_seeds[:split]
test_seeds  = all_seeds[split:]


train_ds = FibonacciModDatasetSmallShample(
    num_samples=25000,
    mod=vocab_size,
    seq_len=seq_len,
    seeds=train_seeds
)

test_ds = FibonacciModDatasetSmallShample(
    num_samples=5000,
    mod=vocab_size,
    seq_len=seq_len,
    seeds=test_seeds
)


train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

epoch = 200
total_accuray = 0 

model = MinimalTransformer(vocab_size=vocab_size).to(device)
print(model.d_model)
file_name = "dim_4_layer_3_alterset.pth"

train_model(model, train_loader, epochs=epoch, test_loader=test_loader, decay=0.01)
avg_loss, eval_accuracy  = evaluate_model(model, test_loader, show_accuracy=True)

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'train_loss_history': train_plot,
    'eval_loss_history': eval_plot,
    'epoch': epoch,
    'total_accuracy': eval_accuracy, 
}
torch.save(checkpoint, full_path)
print(f"Successfully saved to: {full_path}")


8
Epoch 1, Loss (Training): 1.9998 Loss(val): 1.6973
Epoch 2, Loss (Training): 1.6160 Loss(val): 1.6767
Epoch 3, Loss (Training): 1.6083 Loss(val): 1.6822
Epoch 4, Loss (Training): 1.5970 Loss(val): 1.6771
Epoch 5, Loss (Training): 1.5764 Loss(val): 1.6711
Epoch 6, Loss (Training): 1.5716 Loss(val): 1.6666
Epoch 7, Loss (Training): 1.5702 Loss(val): 1.6789
Epoch 8, Loss (Training): 1.5686 Loss(val): 1.6807
Epoch 9, Loss (Training): 1.5673 Loss(val): 1.6968
Epoch 10, Loss (Training): 1.5663 Loss(val): 1.6993
Epoch 11, Loss (Training): 1.5641 Loss(val): 1.7291
Epoch 12, Loss (Training): 1.5626 Loss(val): 1.7282
Epoch 13, Loss (Training): 1.5599 Loss(val): 1.7314
Epoch 14, Loss (Training): 1.5543 Loss(val): 1.7096
Epoch 15, Loss (Training): 1.5451 Loss(val): 1.7513
Epoch 16, Loss (Training): 1.5330 Loss(val): 1.6569
Epoch 17, Loss (Training): 1.5227 Loss(val): 1.6660
Epoch 18, Loss (Training): 1.5072 Loss(val): 1.6974
Epoch 19, Loss (Training): 1.4960 Loss(val): 1.6923
Epoch 20, Loss (Tra